# Tabular Foundation Models: How TabPFN Predicts Without Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/tabular_foundation_models.ipynb)

Companion notebook to [Tabular Foundation Models: How TabPFN Predicts Without Training](https://sesen.ai/blog/tabular-foundation-models-tabpfn).

TabPFN is a transformer pre-trained once on millions of synthetic tabular datasets. To predict on your data you do not train anything: you pass labelled train rows and unlabelled test rows together, and it labels the test rows in a single forward pass (in-context learning).

Here we (1) use it in three lines, (2) watch its decision boundary form as we add in-context examples, and (3) benchmark it against logistic regression and gradient boosting.

> The blog post also benchmarks XGBoost and HistGradientBoosting. We leave them out of this notebook because, on macOS, importing TabPFN (torch) alongside XGBoost in one kernel can deadlock via conflicting OpenMP runtimes. Scikit-learn's `GradientBoostingClassifier` has no such conflict, so we use it as the tree baseline here.

## Setup

In [ ]:
# Uncomment on Colab / fresh environments
# !pip install tabpfn scikit-learn matplotlib numpy

In [ ]:
import time, warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tabpfn import TabPFNClassifier
print('TabPFN ready')

## Three lines, no training loop

`fit` does not optimise anything: it stores the training rows so they can be fed into the transformer alongside the test rows at prediction time. The pre-trained weights never change.

In [ ]:
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

clf = TabPFNClassifier(device='cpu')
clf.fit(Xtr, ytr)            # stores data; no gradient descent
print('accuracy:', round(clf.score(Xte, yte), 4))

## In-context learning: the boundary forms in forward passes

We fit TabPFN on a growing number of in-context examples and plot its decision boundary each time. There are no training epochs between plots; every boundary is a single forward pass over the points shown.

In [ ]:
from sklearn.datasets import make_moons
from matplotlib.colors import ListedColormap
Xm, ym = make_moons(n_samples=400, noise=0.3, random_state=0)
perm = np.random.default_rng(0).permutation(len(Xm)); Xm, ym = Xm[perm], ym[perm]
pad, G = 0.5, 80
xx, yy = np.meshgrid(np.linspace(Xm[:,0].min()-pad, Xm[:,0].max()+pad, G),
                     np.linspace(Xm[:,1].min()-pad, Xm[:,1].max()+pad, G))
grid = np.c_[xx.ravel(), yy.ravel()]
cm_pt = ListedColormap(['#dc2626', '#2563eb'])

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, n in zip(axes, [10, 30, 80, 300]):
    clf = TabPFNClassifier(device='cpu'); clf.fit(Xm[:n], ym[:n])
    proba = clf.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, proba, levels=20, cmap='RdBu', alpha=0.6, vmin=0, vmax=1)
    ax.contour(xx, yy, proba, levels=[0.5], colors='k', linewidths=1.5)
    ax.scatter(Xm[:n,0], Xm[:n,1], c=ym[:n], cmap=cm_pt, edgecolor='white', s=25)
    ax.set_title(f'{n} in-context examples'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Benchmark: no training, still competitive

Three methods, five small datasets, three random splits each, all out of the box with no tuning. LogReg and gradient boosting train normally; TabPFN just does its forward pass.

In [ ]:
from sklearn.datasets import load_wine, load_iris, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def synth(n, d, k, sep): return make_classification(n_samples=n, n_features=d,
    n_informative=max(2, d//2), n_redundant=d//4, n_classes=k, class_sep=sep, random_state=0)

datasets = {
    'breast_cancer': load_breast_cancer(return_X_y=True),
    'wine': load_wine(return_X_y=True),
    'iris': load_iris(return_X_y=True),
    'synth_overlap': synth(800, 20, 3, 0.8),
    'synth_wide': synth(500, 50, 2, 1.0),
}

def make(name):
    if name == 'LogReg':   return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    if name == 'GradBoost': return GradientBoostingClassifier(random_state=0)
    if name == 'TabPFN':   return TabPFNClassifier(device='cpu')

rows = {}
for dname, (X, y) in datasets.items():
    X, y = np.asarray(X, float), np.asarray(y)
    rows[dname] = {}
    for mname in ['LogReg', 'GradBoost', 'TabPFN']:
        accs = []
        for s in [0, 1, 2]:
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=s, stratify=y)
            m = make(mname); m.fit(Xtr, ytr); accs.append(accuracy_score(yte, m.predict(Xte)))
        rows[dname][mname] = np.mean(accs)
    print(f"{dname:14s} " + '  '.join(f'{k} {v:.3f}' for k, v in rows[dname].items()), flush=True)

means = {m: np.mean([rows[d][m] for d in datasets]) for m in ['LogReg', 'GradBoost', 'TabPFN']}
print('\nMEAN ', '  '.join(f'{k} {v:.3f}' for k, v in means.items()))

## What to remember

- TabPFN does **in-context learning**: train + test rows go in together, predictions come out in one forward pass, no per-dataset optimisation.
- It is a **Prior-data Fitted Network**: pre-trained once on millions of synthetic datasets to approximate the Bayesian posterior predictive distribution.
- It is strong on **small** tabular data (up to ~10k rows in v2) and wants a GPU beyond a thousand rows. For very large datasets, gradient-boosted trees still win.

## Exercises

1. **Add XGBoost** (in a *separate* process or kernel on macOS) and compare. Does heavy tuning of XGBoost close the gap to untuned TabPFN?
2. **Push the size limit.** Grow `synth` to 5,000 and 20,000 rows. Where does TabPFN's accuracy or runtime stop being practical on CPU?
3. **Inspect calibration.** Use `predict_proba` and a reliability diagram to compare TabPFN's probability calibration against gradient boosting.
4. **Regression.** TabPFN v2 added a `TabPFNRegressor`. Benchmark it against gradient-boosted regression on a small regression dataset.
5. **Corrupt the features.** Add noise columns or missing values and see which method degrades fastest.